# 04 - Framing Extraction Analysis

This notebook analyses the framing dimensions extracted from 400 articles using Ollama + `llama3.2:3b`.
The extraction runs on the full article corpus and stores results in SQLite. This notebook reads those
results and aggregates them to answer: **who does each outlet blame, who do they portray as the victim,
and what solution do they imply?**

The three dimensions map directly to Entman's (1993) framing theory:
- **Villain** - causal interpretation (who caused the problem?)
- **Victim** - moral evaluation (who is harmed?)
- **Solution** - treatment recommendation (what should be done?)

**Important caveat**: all values here are LLM-generated interpretations, not ground-truth labels.
`llama3.2:3b` produced 0 parse failures on this corpus (391/391 valid JSON), but the extracted phrases
reflect the model's reading of each article - not a human annotator's. Treat the outputs as directional
signals, not precise measurements.

**Topics analysed**: Ukraine war (84 articles), US economy (88 articles), climate change (47 articles)

## 1. Setup

In [1]:
import sys
import os

# Add the project root to sys.path so I can import from src/
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.db import get_connection

conn = get_connection()

total = conn.execute("SELECT COUNT(*) FROM articles").fetchone()[0]
parsed = conn.execute("SELECT COUNT(*) FROM articles WHERE framing_parsed = 1").fetchone()[0]
failed = conn.execute("SELECT COUNT(*) FROM articles WHERE framing_parsed = 0").fetchone()[0]

print(f"Total articles in DB:    {total}")
print(f"Framing parsed (success): {parsed}")
print(f"Framing failed:          {failed}")

Total articles in DB:    472
Framing parsed (success): 400
Framing failed:          70


## 2. Framing Coverage Overview

Before looking at specific framing values, I check how many articles have each dimension filled in.
The solution dimension is expected to be sparse - many news articles describe problems without
proposing a fix. Villain and victim should be much fuller.

In [2]:
# Count non-null values for each framing dimension, broken down by topic
# Only look at the three named topics (skip blank topic = GDELT articles with no assigned topic)
rows = conn.execute("""
    SELECT
        topic,
        COUNT(*) as total,
        SUM(CASE WHEN framing_villain  IS NOT NULL THEN 1 ELSE 0 END) as villain_filled,
        SUM(CASE WHEN framing_victim   IS NOT NULL THEN 1 ELSE 0 END) as victim_filled,
        SUM(CASE WHEN framing_solution IS NOT NULL THEN 1 ELSE 0 END) as solution_filled
    FROM articles
    WHERE framing_parsed = 1 AND topic != ''
    GROUP BY topic
    ORDER BY total DESC
""").fetchall()

coverage = pd.DataFrame(rows, columns=["topic", "total", "villain_filled", "victim_filled", "solution_filled"])

# Add fill rate columns so the sparsity of 'solution' is immediately visible
for dim in ["villain", "victim", "solution"]:
    coverage[f"{dim}_pct"] = (coverage[f"{dim}_filled"] / coverage["total"] * 100).round(1)

print(coverage[["topic", "total", "villain_pct", "victim_pct", "solution_pct"]].to_string(index=False))

         topic  total  villain_pct  victim_pct  solution_pct
    US economy     88         96.6        92.0          21.6
   Ukraine war     84         96.4        88.1          20.2
climate change     47         91.5        89.4          17.0


## 3. Helper Functions

Two reusable functions I'll call for each topic:
- `top_framing()` - queries the top N values for a given dimension and topic
- `framing_bar_chart()` - turns that into a horizontal bar chart

In [3]:
def top_framing(topic: str, dimension: str, n: int = 10) -> pd.DataFrame:
    """
    Return the top N most frequent framing values for a given dimension and topic.
    dimension must be one of: 'villain', 'victim', 'solution'.
    """
    col = f"framing_{dimension}"
    rows = conn.execute(f"""
        SELECT {col} as value, COUNT(*) as n
        FROM articles
        WHERE framing_parsed = 1
          AND topic = ?
          AND {col} IS NOT NULL
          AND LOWER({col}) NOT IN ('none', 'null', 'n/a')
        GROUP BY {col}
        ORDER BY n DESC
        LIMIT ?
    """, (topic, n)).fetchall()
    return pd.DataFrame(rows, columns=["value", "count"])


def framing_bar_chart(df: pd.DataFrame, title: str, colour: str) -> go.Figure:
    """
    Horizontal bar chart for framing frequency data.
    Horizontal layout keeps long phrases readable.
    """
    # Reverse order so the highest count appears at the top
    df_plot = df.iloc[::-1].copy()

    fig = go.Figure(go.Bar(
        x=df_plot["count"],
        y=df_plot["value"],
        orientation="h",
        marker_color=colour,
    ))
    fig.update_layout(
        title=title,
        xaxis_title="Number of articles",
        yaxis_title="",
        height=max(300, len(df) * 40),
        width=750,
        margin=dict(l=200),
    )
    return fig

## 4. Ukraine War - Framing Analysis

84 articles. This topic has the clearest expected villain structure - Russia and its government
feature prominently, but the specific phrasing varies ("Russia", "Russian government", "Vladimir Putin").
This is a known limitation of raw LLM extraction: semantically equivalent phrases are counted separately
because there is no normalisation step. The real count for "Russia" as villain is higher than any
single phrase suggests.

In [4]:
ukraine_villains = top_framing("Ukraine war", "villain")
print("Top villains - Ukraine war")
print(ukraine_villains.to_string(index=False))

Top villains - Ukraine war
                        value  count
                       Russia     10
                         Iran      7
global economic uncertainties      3
           Russian government      3
               Vladimir Putin      2
          Russia's government      2
              President Trump      2
                 Keir Starmer      2
                        China      2
               war in Ukraine      1


In [5]:
framing_bar_chart(ukraine_villains, "Ukraine War - Top Villains", "#e15759")

In [6]:
ukraine_victims = top_framing("Ukraine war", "victim")
print("Top victims - Ukraine war")
print(ukraine_victims.to_string(index=False))

Top victims - Ukraine war
                           value  count
             coastal communities      8
             Ukrainian civilians      8
                         Ukraine      5
                   United States      3
Ukrainian civilians and children      2
                    Labour Party      2
          working class citizens      1
               opposition groups      1
                          no one      1
          most vulnerable people      1


In [7]:
framing_bar_chart(ukraine_victims, "Ukraine War - Top Victims", "#4e79a7")

In [8]:
# Solution is sparse (~20% fill rate) - use n=5 instead of 10
ukraine_solutions = top_framing("Ukraine war", "solution", n=5)
print("Top solutions - Ukraine war")
print(ukraine_solutions.to_string(index=False))

Top solutions - Ukraine war
                      value  count
      free trade agreements      2
strengthening China-US ties      1
                  sanctions      1
     pipeline exports surge      1
              none provided      1


In [9]:
framing_bar_chart(ukraine_solutions, "Ukraine War - Top Solutions (sparse)", "#76b7b2")

### 4a. Ukraine War - Outlet-Level Villain Table

This is where framing theory becomes most visible: does Fox News assign a different villain than BBC
for the same conflict? I filter to outlets with at least 2 articles on this topic so single-article
outlets do not distort the picture, then show each outlet's most frequently assigned villain.

In [10]:
def outlet_villain_table(topic: str, min_articles: int = 2) -> pd.DataFrame:
    """
    For each outlet with at least min_articles on this topic, find the most
    frequently assigned villain. Ties are broken by alphabetical order.
    """
    # Step 1: count articles per outlet for this topic (with framing parsed and villain present)
    outlet_counts = conn.execute("""
        SELECT outlet, COUNT(*) as n
        FROM articles
        WHERE framing_parsed = 1
          AND topic = ?
          AND framing_villain IS NOT NULL
          AND LOWER(framing_villain) NOT IN ('none', 'null', 'n/a')
        GROUP BY outlet
        HAVING COUNT(*) >= ?
        ORDER BY n DESC
    """, (topic, min_articles)).fetchall()

    qualified_outlets = [r[0] for r in outlet_counts]
    outlet_article_count = {r[0]: r[1] for r in outlet_counts}

    if not qualified_outlets:
        print(f"No outlets with >= {min_articles} articles on topic '{topic}'")
        return pd.DataFrame()

    # Step 2: for each qualified outlet, find its top villain
    rows = []
    for outlet in qualified_outlets:
        result = conn.execute("""
            SELECT framing_villain, COUNT(*) as n
            FROM articles
            WHERE framing_parsed = 1
              AND topic = ?
              AND outlet = ?
              AND framing_villain IS NOT NULL
              AND LOWER(framing_villain) NOT IN ('none', 'null', 'n/a')
            GROUP BY framing_villain
            ORDER BY n DESC
            LIMIT 1
        """, (topic, outlet)).fetchone()

        if result:
            rows.append({
                "outlet":         outlet,
                "articles":       outlet_article_count[outlet],
                "top_villain":    result[0],
                "villain_count":  result[1],
            })

    return pd.DataFrame(rows)


ukraine_outlet_table = outlet_villain_table("Ukraine war", min_articles=2)
print(f"Outlets with >= 2 Ukraine war articles: {len(ukraine_outlet_table)}")
print()
print(ukraine_outlet_table.to_string(index=False))

Outlets with >= 2 Ukraine war articles: 14

            outlet  articles                            top_villain  villain_count
  freerepublic_com         8 extremist Christian dispensationalists              1
the_times_of_india         5          global economic uncertainties              2
          cbc_news         4              conflicting global powers              1
        al_jazeera         4                          US and Israel              1
        dw_english         3              US President Donald Trump              1
   the_irish_times         2                                 Russia              1
      the_diplomat         2                                 Russia              1
               rte         2                                 Russia              2
      oilprice_com         2                       Ukrainian drones              1
         europa_eu         2               global economic activity              1
     dailymail_com         2               

## 5. US Economy - Framing Analysis

88 articles - the largest named topic in the corpus. The corpus is dominated by Trump-Xi trade summit
coverage and Federal Reserve / Warsh confirmation stories (see notebook 03). I expect the villain
dimension to split between China, Trump administration policy, and economic forces.

In [11]:
econ_villains = top_framing("US economy", "villain")
print("Top villains - US economy")
print(econ_villains.to_string(index=False))

Top villains - US economy
                      value  count
                      China     10
                       Iran      5
                  Hezbollah      3
federal government inaction      2
                 Xi Jinping      2
                         US      2
      wooden boat operators      1
                   viewbots      1
           unknown entities      1
                    unknown      1


In [12]:
framing_bar_chart(econ_villains, "US Economy - Top Villains", "#e15759")

In [13]:
econ_victims = top_framing("US economy", "victim")
print("Top victims - US economy")
print(econ_victims.to_string(index=False))

Top victims - US economy
                                       value  count
                         coastal communities     11
                                      Taiwan      4
                                        Iran      4
                                  US economy      2
                            Taiwanese people      2
                          Lebanese civilians      2
                                       India      2
working parents with caring responsibilities      1
                           working Americans      1
                                     workers      1


In [14]:
framing_bar_chart(econ_victims, "US Economy - Top Victims", "#4e79a7")

In [15]:
econ_solutions = top_framing("US economy", "solution", n=5)
print("Top solutions - US economy")
print(econ_solutions.to_string(index=False))

Top solutions - US economy
                                value  count
               value-added production      1
traditional interest rate adjustments      1
                technology conversion      1
               slash welfare spending      1
                 reader contributions      1


In [16]:
framing_bar_chart(econ_solutions, "US Economy - Top Solutions (sparse)", "#76b7b2")

In [17]:
econ_outlet_table = outlet_villain_table("US economy", min_articles=2)
print(f"Outlets with >= 2 US economy articles: {len(econ_outlet_table)}")
print()
print(econ_outlet_table.to_string(index=False))

Outlets with >= 2 US economy articles: 15

                outlet  articles                top_villain  villain_count
    the_times_of_india         8        rising crude prices              1
      freerepublic_com         6   artificial turf industry              1
       the_irish_times         4 rigid workplace structures              1
       naturalnews_com         4                       Iran              2
         dailymail_com         4                 Xi Jinping              1
       crypto_briefing         4    crypto industry players              1
                   rte         3            President Trump              1
         khabarhub_com         3      major external powers              1
        independent_ie         3                      China              2
         thejournal_ie         2                    unknown              1
                    rt         2                       Iran              1
         new_york_post         2                       Ir

## 6. Climate Change - Framing Analysis

47 articles - the smallest topic, and the one most affected by topic contamination (notebook 03
showed Eurovision and lifestyle articles appearing in cluster results). The framing values here
will reflect that contamination - some extracted villains and victims will have nothing to do
with climate. I note this rather than filtering it out, since it shows a real data quality issue.

In [18]:
climate_villains = top_framing("climate change", "villain")
print("Top villains - climate change")
print(climate_villains.to_string(index=False))

Top villains - climate change
                                  value  count
                         climate change      3
white supremacists and MAGA Republicans      1
               third-party manufacturer      1
                   negative stereotypes      1
                          many churches      1
         human generated CO 2 emissions      1
      global warming and climate change      1
                         global warming      1
                           fossil fuels      1
           fertiliser and pesticide use      1


In [19]:
framing_bar_chart(climate_villains, "Climate Change - Top Villains", "#e15759")

In [20]:
climate_victims = top_framing("climate change", "victim")
print("Top victims - climate change")
print(climate_victims.to_string(index=False))

Top victims - climate change
                                value  count
                     players and fans      3
                  coastal communities      3
                          environment      2
                   the general public      1
                        the art world      1
                   the Jackson Estate      1
                              players      1
            peanut-allergic consumers      1
patients with weakened immune systems      1
                    mycorrhizal fungi      1


In [21]:
framing_bar_chart(climate_victims, "Climate Change - Top Victims", "#4e79a7")

In [22]:
climate_solutions = top_framing("climate change", "solution", n=5)
print("Top solutions - climate change")
print(climate_solutions.to_string(index=False))

Top solutions - climate change
                                   value  count
                    values-driven policy      1
                 urban planning policies      1
restoration projects with native strains      1
           proactive disaster management      1
                        performing music      1


In [23]:
framing_bar_chart(climate_solutions, "Climate Change - Top Solutions (sparse)", "#76b7b2")

In [24]:
climate_outlet_table = outlet_villain_table("climate change", min_articles=2)
print(f"Outlets with >= 2 climate change articles: {len(climate_outlet_table)}")
print()
print(climate_outlet_table.to_string(index=False))

Outlets with >= 2 climate change articles: 3

              outlet  articles                    top_villain  villain_count
 the_independent_com        17       third-party manufacturer              1
       science_daily         2 human generated CO 2 emissions              1
peoplesreview_com_np         2                  US government              1


## 7. Cross-Topic Villain Comparison

A side-by-side table of the top 5 villains per topic. This makes it easy to see how the villain
structure differs across stories - a useful summary for the dashboard Framing Explorer page.

In [25]:
# Pull top 5 villains for each topic and stitch into a comparison table
topics = {
    "Ukraine war":    top_framing("Ukraine war",   "villain", n=5),
    "US economy":     top_framing("US economy",    "villain", n=5),
    "climate change": top_framing("climate change","villain", n=5),
}

# Pad shorter lists to 5 rows so the table is uniform
comparison_rows = []
for rank in range(5):
    row = {"rank": rank + 1}
    for topic_name, df in topics.items():
        if rank < len(df):
            row[topic_name] = f"{df.iloc[rank]['value']} ({df.iloc[rank]['count']})"
        else:
            row[topic_name] = "-"
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index("rank")
print("Top 5 villains per topic (count in parentheses)")
print()
print(comparison_df.to_string())

Top 5 villains per topic (count in parentheses)

                            Ukraine war                       US economy                               climate change
rank                                                                                                                 
1                           Russia (10)                       China (10)                           climate change (3)
2                              Iran (7)                         Iran (5)  white supremacists and MAGA Republicans (1)
3     global economic uncertainties (3)                    Hezbollah (3)                 third-party manufacturer (1)
4                Russian government (3)  federal government inaction (2)                     negative stereotypes (1)
5                    Vladimir Putin (2)                   Xi Jinping (2)                            many churches (1)


## 8. Framing Dimensions Summary Chart

A grouped bar chart showing fill rates per dimension per topic. This goes in the README to show
that villain and victim are well-populated while solution is structurally sparse.

In [26]:
# Re-use the coverage DataFrame from section 2
topics_ordered = ["US economy", "Ukraine war", "climate change"]
coverage_plot = coverage[coverage["topic"].isin(topics_ordered)].set_index("topic").loc[topics_ordered]

fig = go.Figure()

dim_colours = {"villain": "#e15759", "victim": "#4e79a7", "solution": "#76b7b2"}

for dim, colour in dim_colours.items():
    fig.add_trace(go.Bar(
        name=dim,
        x=topics_ordered,
        y=coverage_plot[f"{dim}_pct"],
        marker_color=colour,
    ))

fig.update_layout(
    barmode="group",
    title="Framing Dimension Fill Rate by Topic (%)",
    xaxis_title="Topic",
    yaxis_title="Fill rate (%)",
    yaxis_range=[0, 110],
    legend_title="Dimension",
    width=700,
    height=400,
)
fig

## 9. Observations

**Villain dimension is the clearest signal**: Fill rates are 95%+ across all topics. The extracted
villains are coherent and topic-specific - Russia/Putin for Ukraine, tariffs/trade policy for US
economy. This supports the pre-analysis hypothesis that villain framing would be the most consistent
dimension.

**Phrase fragmentation is a real problem**: "Russia", "Russian government", "Russia's government",
and "Vladimir Putin" are semantically the same villain but counted separately. The true frequency of
Russia-as-villain in Ukraine coverage is higher than any single phrase shows. A normalisation step
(entity linking or simple fuzzy matching) would strengthen this analysis. This is acknowledged as a
limitation rather than fixed here - the raw LLM output is the source of truth for this pipeline.

**Solution dimension is structurally sparse**: 17-22% fill rate is expected, not a bug. News
reporting describes events and assigns blame far more often than it recommends solutions. The low
fill rate reflects editorial norms, not extraction failure.

**Topic contamination carries through to framing**: The climate change cluster in notebook 03
contained Eurovision and lifestyle articles. Those same off-topic articles produce off-topic framing
values here (e.g. music industry villains appearing in climate change results). A pre-filtering step
before framing extraction would improve signal quality, but was not applied - the extraction ran on
all articles regardless of topical relevance.

**Outlet-level villain analysis is limited by sample size**: With most outlets contributing only
2-5 articles per topic, per-outlet villain distributions are noisy. The outlet table is directionally
interesting but should not be over-interpreted. A larger corpus - achievable with more ingestion runs
or a broader GDELT pull - would make this analysis more reliable.

**Hypothesis check (partial)**: The pre-analysis hypothesis was that villain framing would show the
strongest outlet-level consistency. This is directionally supported - Russia is the dominant villain
in Ukraine coverage across nearly all outlets. The US economy villain picture is more fragmented,
which is also consistent with the hypothesis (economic topics have more distributed blame than
military conflicts). A larger corpus is needed to test this rigorously.